# Domain-Adaptive Pretraining (TAPT / MLM) for mmBERT
### Target: Infuse LGBTQIA+ Nuances, Social Media Slang, & Subtle Hate Knowledge
**Backbone Model**: `jhu-clsp/mmbert-base`  
**Pretraining Objective**: Masked Language Modeling (MLM 15%)  
**Corpus**: Filtered LGBTQ+ data from `measuring-hate-speech`, `toxigen-data`, `tides`, and in-domain YouTube context.

In [ ]:
# -----------------------------------------------------------------------------
# 0. CONFIGURATION & HUGGING FACE AUTHENTICATION (OPTIONAL)
# -----------------------------------------------------------------------------
import os
# Must be set before importing torch or initializing CUDA to force single GPU and prevent DataParallel bug
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import gc
import glob
import torch
import pandas as pd
import numpy as np

# Set seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

BASE_MODEL_NAME = "jhu-clsp/mmbert-base"
OUTPUT_LOCAL_DIR = "./mmbert-queer-hate-adapted"

# Optional: Set your Hugging Face credentials if you want to push to Hub directly
# Set to None if you only want to save locally in Kaggle output
HF_WRITE_TOKEN = None  # e.g. "hf_xxxxxxxxxxxxxxxxxxxx"
HF_REPO_NAME = None    # e.g. "your-username/mmbert-queer-hate-adapted"

EPOCHS = 4
BATCH_SIZE = 16            # Reduced to prevent OOM with large vocab size
GRAD_ACCUM_STEPS = 2       # Effective batch size = 16 * 2 = 32
MAX_LENGTH = 128           # Optimal for comments & short context
LEARNING_RATE = 5e-5
MLM_PROBABILITY = 0.15
UNFREEZE_TOP_N_LAYERS = 6  # Train only top 6 layers + MLM head to save memory & retain grammar

if HF_WRITE_TOKEN and HF_WRITE_TOKEN.startswith("hf_"):
    from huggingface_hub import login
    login(token=HF_WRITE_TOKEN)
    print("✅ Logged in to Hugging Face Hub.")
else:
    print("ℹ️ Running in local mode (Model will be saved to Kaggle output).")

In [ ]:
# -----------------------------------------------------------------------------
# 1. LOAD AND FILTER DOMAIN DATASETS
# -----------------------------------------------------------------------------
from datasets import load_dataset, Dataset

collected_texts = []
print("==================== COLLECTING DOMAIN TEXTS ====================")

# --- Dataset A: Measuring Hate Speech (UC Berkeley D-Lab) ---
try:
    print("\n[1/4] Loading 'ucberkeley-dlab/measuring-hate-speech'...")
    mhs_ds = load_dataset("ucberkeley-dlab/measuring-hate-speech", split="train")
    mhs_df = mhs_ds.to_pandas()
    
    cond = (
        (mhs_df.get('target_sexuality', False) == True) |
        (mhs_df.get('target_gender', False) == True) |
        (mhs_df.get('hate_speech_score', 0) > 0.0)
    )
    mhs_filtered = mhs_df[cond]['text'].dropna().astype(str).tolist()
    print(f"  --> Extracted {len(mhs_filtered)} LGBTQ+ & hate speech sentences from MHS.")
    collected_texts.extend(mhs_filtered)
except Exception as e:
    print(f"  ⚠️ Error loading MHS: {e}")

# --- Dataset B: ToxiGen (Subtle / Implicit Toxicity) ---
try:
    print("\n[2/4] Loading 'toxigen/toxigen-data' (annotated split)...")
    tox_ds = load_dataset("toxigen/toxigen-data", name="annotated", split="train")
    tox_df = tox_ds.to_pandas()
    
    if 'target_group' in tox_df.columns:
        mask = tox_df['target_group'].astype(str).str.contains('lgbtq|gay|trans|lesbian|queer', case=False, na=False)
        tox_filtered = tox_df[mask]['text'].dropna().astype(str).tolist()
    else:
        tox_filtered = tox_df['text'].dropna().astype(str).tolist()[:20000]
    print(f"  --> Extracted {len(tox_filtered)} subtle/implicit texts from ToxiGen.")
    collected_texts.extend(tox_filtered)
except Exception as e:
    print(f"  ⚠️ Error loading ToxiGen: {e}")

# --- Dataset C: TIDES (Targeted Identity Disparagement & Exclusion) ---
try:
    print("\n[3/4] Loading 'falseiftrue/tides'...")
    tides_ds = load_dataset("falseiftrue/tides", split="train")
    tides_df = tides_ds.to_pandas()
    
    text_col = 'text' if 'text' in tides_df.columns else tides_df.columns[0]
    tides_filtered = tides_df[text_col].dropna().astype(str).tolist()
    print(f"  --> Extracted {len(tides_filtered)} texts from TIDES.")
    collected_texts.extend(tides_filtered)
except Exception as e:
    print(f"  ⚠️ Error loading TIDES: {e}")

# --- Dataset D: In-Domain YouTube Dataset (StereoQueer) ---
print("\n[4/4] Searching for In-Domain StereoQueer files in Kaggle / Local paths...")
possible_paths = [
    "/kaggle/input/**/*.tsv",
    "/kaggle/input/**/*.csv",
    "data/*.tsv",
    "data/*.csv",
    "../LGBT/*.tsv",
    "LGBT/*.tsv",
    "*.tsv",
    "*.csv"
]

in_domain_count = 0
found_files = sorted({f for pat in possible_paths for f in glob.glob(pat, recursive=True)})
for filepath in found_files:
    if not os.path.isfile(filepath) or os.path.basename(filepath).startswith('.'):
        continue
    try:
        sep = '\t' if filepath.endswith('.tsv') else ','
        try:
            df_in = pd.read_csv(filepath, sep=sep, quoting=1, on_bad_lines='skip')
        except Exception:
            try:
                df_in = pd.read_csv(filepath, sep=None, engine='python', on_bad_lines='skip')
            except Exception:
                df_in = pd.read_csv(filepath, on_bad_lines='skip')
        file_added = 0
        # 1. Add compound structured sequences if yt_comment exists
        if 'yt_comment' in df_in.columns:
            comments = df_in['yt_comment'].fillna('').astype(str)
            titles = df_in['yt_title'].fillna('').astype(str) if 'yt_title' in df_in.columns else pd.Series(['']*len(df_in))
            descs = df_in['yt_description'].fillna('').astype(str) if 'yt_description' in df_in.columns else pd.Series(['']*len(df_in))
            compound_texts = [
                f"comment: {c} [SEP] title: {t} [SEP] desc: {d}"
                for c, t, d in zip(comments, titles, descs) if len(c.strip()) > 3
            ]
            collected_texts.extend(compound_texts)
            file_added += len(compound_texts)
        # 2. Also add standalone individual sentences
        for col in ['yt_comment', 'yt_title', 'yt_description', 'comment', 'text', 'body']:
            if col in df_in.columns:
                texts = df_in[col].dropna().astype(str).tolist()
                collected_texts.extend(texts)
                file_added += len(texts)
        if file_added > 0:
            print(f"  --> Added {file_added} sentences from: {filepath}")
            in_domain_count += file_added
    except Exception as err:
        print(f"  ⚠️ Could not read {filepath}: {err}")

# Clean & Deduplicate
cleaned_texts = []
seen = set()
for t in collected_texts:
    clean_t = t.strip()
    if len(clean_t) >= 12 and clean_t not in seen:
        seen.add(clean_t)
        cleaned_texts.append(clean_t)

print(f"\n✨ TOTAL UNIQUE CURATED SENTENCES FOR PRETRAINING: {len(cleaned_texts):,}")

In [ ]:
# -----------------------------------------------------------------------------
# 2. TOKENIZATION & DYNAMIC 15% MASKING COLLATOR
# -----------------------------------------------------------------------------
from transformers import AutoTokenizer, AutoModelForMaskedLM, DataCollatorForLanguageModeling

print(f"Loading base tokenizer & MLM model from: {BASE_MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
model = AutoModelForMaskedLM.from_pretrained(BASE_MODEL_NAME)

# Freeze bottom layers to prevent OOM and preserve core multilingual grammar
if UNFREEZE_TOP_N_LAYERS is not None:
    for param in model.parameters():
        param.requires_grad = False
    # Unfreeze MLM head
    for head_attr in ['head', 'lm_head', 'cls']:
        if hasattr(model, head_attr):
            for p in getattr(model, head_attr).parameters():
                p.requires_grad = True
    # Unfreeze top N transformer layers
    encoder = getattr(model, 'model', getattr(model, 'modernbert', model))
    layers = getattr(encoder, 'layers', None)
    if layers is not None:
        total_l = len(layers)
        for layer in layers[max(0, total_l - UNFREEZE_TOP_N_LAYERS):]:
            for p in layer.parameters():
                p.requires_grad = True
        print(f"❄️ Froze {total_l - UNFREEZE_TOP_N_LAYERS} bottom layers. 🔥 Training top {UNFREEZE_TOP_N_LAYERS} layers + MLM Head.")

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable Parameters: {trainable_params:,} / {total_params:,} ({trainable_params/total_params*100:.1f}%)")

raw_dataset = Dataset.from_dict({"text": cleaned_texts})

def tokenize_fn(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        return_special_tokens_mask=True
    )

print("Tokenizing dataset...")
tokenized_dataset = raw_dataset.map(
    tokenize_fn,
    batched=True,
    num_proc=2,
    remove_columns=["text"]
)

# Dynamic Masked Language Modeling Collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=MLM_PROBABILITY
)
print("✅ Tokenization and DataCollator configured.")

In [ ]:
# -----------------------------------------------------------------------------
# 3. TRAINER & PRETRAINING LOOP WITH REAL-TIME LOSS LOGGING
# -----------------------------------------------------------------------------
import math
import time
from transformers import Trainer, TrainingArguments, TrainerCallback

class MLMLoggingCallback(TrainerCallback):
    def __init__(self):
        self.start_time = None
        
    def on_train_begin(self, args, state, control, **kwargs):
        self.start_time = time.time()
        print("\n" + "="*82)
        print(f"🚀 [MLM WARMUP TRAINING STARTED] Epochs: {args.num_train_epochs} | Total Steps: {state.max_steps}")
        print("="*82)
        print(f"{'Step':<11} | {'Epoch':<7} | {'MLM Loss':<10} | {'Perplexity':<12} | {'LR':<10} | {'GPU Mem':<9} | {'Elapsed':<8}")
        print("-"*82)

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None:
            return
        loss = logs.get("loss", None)
        lr = logs.get("learning_rate", None)
        epoch = logs.get("epoch", state.epoch if state else 0.0)
        step = state.global_step
        max_step = state.max_steps
        
        if loss is not None:
            ppl = math.exp(min(loss, 20.0))
            mem = f"{torch.cuda.memory_allocated()/1e9:.1f}GB" if torch.cuda.is_available() else "CPU"
            elapsed = f"{time.time() - self.start_time:.0f}s" if self.start_time else "0s"
            lr_str = f"{lr:.2e}" if lr is not None else "N/A"
            print(f"[{step:04d}/{max_step:04d}] | {epoch:5.2f}  | {loss:8.4f}   | {ppl:10.2f}   | {lr_str:<10} | {mem:<9} | {elapsed:<8}")

    def on_epoch_end(self, args, state, control, **kwargs):
        print(f"✨ --- [EPOCH {int(state.epoch)} COMPLETE] ---")

    def on_train_end(self, args, state, control, **kwargs):
        total_time = (time.time() - self.start_time) if self.start_time else 0
        print("="*82)
        print(f"🎉 [WARMUP TRAINING FINISHED] Total Time: {total_time/60:.2f} mins")
        print("="*82 + "\n")

training_args = TrainingArguments(
    output_dir=OUTPUT_LOCAL_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    fp16=torch.cuda.is_available(),
    gradient_checkpointing=True,
    logging_steps=10,             # Log every 10 steps for instant feedback
    save_strategy="epoch",
    save_total_limit=1,
    dataloader_num_workers=2,
    disable_tqdm=False,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=tokenized_dataset,
    callbacks=[MLMLoggingCallback()]
)

print("🚀 Starting Domain-Adaptive MLM Pretraining...")
trainer.train()
print("✅ MLM Pretraining Complete!")

In [ ]:
# -----------------------------------------------------------------------------
# 4. SAVE WEIGHTS & PUSH TO HUGGING FACE (OR KAGGLE OUTPUT)
# -----------------------------------------------------------------------------
print(f"Saving adapted model and tokenizer to: {OUTPUT_LOCAL_DIR}...")
model.save_pretrained(OUTPUT_LOCAL_DIR)
tokenizer.save_pretrained(OUTPUT_LOCAL_DIR)

# Push to Hugging Face if configured
if HF_REPO_NAME and HF_WRITE_TOKEN:
    print(f"Pushing to Hugging Face Hub: {HF_REPO_NAME}...")
    model.push_to_hub(HF_REPO_NAME)
    tokenizer.push_to_hub(HF_REPO_NAME)
    print(f"🎉 Model successfully published to: https://huggingface.co/{HF_REPO_NAME}")
else:
    print("\n📁 Model saved locally in Kaggle notebook output!")
    print(f"Path: {os.path.abspath(OUTPUT_LOCAL_DIR)}")
    print("You can load it in Task B with:")
    print(f"  config.mmbert_model_name = '{OUTPUT_LOCAL_DIR}'")